In [1]:
import numpy as np
import pandas as pd

import spacy
from spacy import displacy
from spacy.language import Language
from spacy.glossary import explain

from typing import Literal

from pprint import pprint
from Node import Node

MARKDOWN

# TO DO

edit equals function to determine node order
edit get time to either 1) use info from step or 2) predefined info
edit get utensil to get utensil from kitchen_utensils.txt


In [ ]:
#returns all keys where the pos var = the find var
def findKey(tokenDict: dict, pos: list[Literal["PartOfSpeech", "HEAD", "Relation"]], find: list):
    ret = []
    for key in tokenDict.keys():
        fullMatch = True
        for i in range(len(pos)):
            #print(f"({tokenDict[key][pos[i]]},{find[i]})")
            
            if tokenDict[key][pos[i]] != find[i]: #match on each index
                fullMatch = False
        
        if fullMatch:
            ret.append(key)
    return ret

def getTime(tokenDict):
    for key in tokenDict.keys():
        if tokenDict[key] in ("seconds", "minutes", "hours"):
            for n in findKey(tokenDict, pos=["HEAD", "Relation"], find=[key, "nummod"]):
                return n   
    return 0

def getUtensil(tokenDict):
    raise Exception

def makeNodes(tokenDict: dict) -> list[Node]:
    node = []
    for root in findKey(tokenDict, pos=["Relation"], find=["ROOT"]): #find all nodes from root
        for verb in findKey(tokenDict, pos=["HEAD", "PartOfSpeech"], find=[root, "VB"]):
            for dObj in findKey(tokenDict, pos=["HEAD", "Relation"], find=[verb, "dobj"]):
                node.append(
                    Node(
                        verb, 
                        dObj, 
                        None, 
                        getTime(tokenDict), 
                        getUtensil(tokenDict)
                    )
                )
    return node

def makeTokenDict(sentence: str, nlp: Language):
    doc = nlp(sentence)
    ret = {}
    #print(f"\nSentence: {sentence}")
    for token in doc:
        # print("Token: {:<18}| Tag: {:<42}| Head: {:<18}| Dep: {:<25}| Pos: {:<20}".format(
        #     token.text, 
        #     (token.tag_), 
        #     token.head.text, 
        #     (token.dep_), 
        #     (token.pos_)
        # ))
        ret.update({token.text: {"PartOfSpeech": token.tag_, "HEAD": token.head.text, "Relation": token.dep_}})
    #displacy.render(doc, style="dep")
    return ret

def equals(dobj1: str, dobj2: str):

    s1 = dobj1.split(" ")
    s2 = dobj2.split(" ")

    for s in s1:
        if s in s2:
            return True
    return False

def makeGraph(steps: list[str], nlp: Language):
    nodes : list[Node] = []

    for step in steps:
        for sentence in step.split("."):
            tokenDict = makeTokenDict(sentence, nlp)
            for node in makeNodes(tokenDict):
                nodes.append(node)

    for i in range(len(nodes)-1):
        for j in range(i, len(nodes)):
            if equals(nodes[i].dobj, nodes[j].dobj):
                nodes[i].nextNode = nodes[j]

    return nodes
        #with nodes, we have to determine sequentiallity
        #if the direct obj of one node is the direct obj of the other, set the other = curr.next



In [ ]:
steps = open("text.txt").readlines()

nlp = spacy.load("en_core_web_trf")
nlp.add_pipe("merge_noun_chunks")


#expects a list where each entry is a string. 
#the string should be each individual step from the json file
#do this with a for loop over json file
graph = makeGraph(steps, nlp)

for node in graph:
    print(node)

#if this is the first one, 


/home/mrow/Projects/DubHacks26/dubvenv/lib/python3.14/site-packages/torch/jit/_script.py:1485: FutureWarning: `torch.jit.script` is not supported in Python 3.14+ and may break. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


verb: Season, dobj: pork chunks, next step: Season pork chunks, time to cook: 0, utensil: pan
verb: distribute, dobj: seasonings, next step: distribute seasonings, time to cook: 0, utensil: pan
verb: Heat, dobj: vegetable oil, next step: Heat vegetable oil, time to cook: 0, utensil: pan
verb: Place, dobj: onions, next step: Place onions, time to cook: 0, utensil: pan
verb: Add, dobj: garlic, next step: Add garlic, time to cook: 0, utensil: pan
verb: Increase, dobj: the heat, next step: raise heat, time to cook: 0, utensil: pan
verb: stir, dobj: mustard, next step: stir mustard, time to cook: 0, utensil: pan
verb: Pour, dobj: cream, next step: Pour cream, time to cook: 0, utensil: pan
verb: add, dobj: more chicken broth, next step: add more chicken broth, time to cook: 0, utensil: pan
verb: Add, dobj: sage, next step: Add sage, time to cook: 0, utensil: pan
verb: Add, dobj: carrots, next step: Add carrots, time to cook: 0, utensil: pan
verb: Add, dobj: peas, next step: Add peas, time to

AttributeError: 'NoneType' object has no attribute 'verb'